In [6]:
# Alex Net and VGG implementation
# Alex net arch
# 5 convolutional 
# 3 fully connected layer
# ReLU Nonlinearity
# 


$$g_t = \nabla_\theta L(\theta_{t-1}) + \lambda\theta_{t-1}$$

$$v_t = \beta v_{t-1} + g_t$$

$$\theta_t = \theta_{t-1} - \eta v_t$$

$$
\boxed{
\theta_t =
\theta_{t-1}
-\eta\left[
\beta v_{t-1}
+\nabla_\theta L(\theta_{t-1})
+\lambda\theta_{t-1}
\right]
}
$$

In [ ]:
# Alex Net
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision import datasets
from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"

class AlexNet(nn.Module):
    def __init__(self):
        super().__init__()
        # N -> number of batches / number of diff samples / number of images
        # Padding(P) 
        # stride(s)
        # kernel size 
        # input - (N, C, H, W) - (N, 3, 227, 227)
        self.net = nn.Sequential(
            nn.Conv2d(3, 96, kernel_size=11, stride=4), # (c_in, c_out, kernel_size, stride) -> (N, 96, 55, 55)
            nn.ReLU(inplace=True),
            nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2),
            nn.MaxPool2d(kernel_size=3, stride=2), # (N, 96, 27, 27)

            nn.Conv2d(96, 256, kernel_size=5, padding=2), # (c_in, c_out, kernel_size, stride) -> (N, 256, 27, 27)
            nn.ReLU(inplace=True),
            nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2),
            nn.MaxPool2d(kernel_size=3, stride=2), # (N, 256, 13, 13),

            nn.Conv2d(256, 384, kernel_size=3, padding=1), # (N, 384, 13, 13)
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 384, kernel_size=3, padding=1), # (N, 384, 13, 13)
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1), # (N, 256, 13, 13)
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2), # (N, 256, 6, 6),

            # now fully connected layer
            # flatten 
            nn.Flatten(), # (N, 9216)
            nn.Dropout(p=0.5),
            nn.Linear(9216, 4096), # (N, 4096)
            nn.ReLU(inplace=True),

            nn.Dropout(p=0.5),
            nn.Linear(4096, 4096), # (N, 4096)
            nn.ReLU(inplace=True),
            # nn.Linear(4096, 1000), #(N, 1000)
            nn.Linear(4096, 10), #(N, 1000)
        )
    def _init_weights(self):
        for m in self.net:
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                # nn.init.normal_(m.weight, mean=0, std=0.01)
                nn.init.normal_(m.weight, mean=0, std=0.01)
                nn.init.constant_(m.bias,  0)
        # paper: bias=1 on conv2, conv4, conv5 and the two hidden FC layers
        for i in (4, 10, 12, 17, 20):
            nn.init.constant_(self.net[i].bias, 1)

    def forward(self, x):
        return self.net(x)

# download (uncomment on a new machine; ~340 MB, extracts to ../data/imagenette2-320)
# from torchvision.datasets.utils import download_and_extract_archive
# download_and_extract_archive("https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz", download_root="../data")

# data loader
norm = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
train_tf = transforms.Compose([transforms.Resize(256), transforms.RandomCrop(227),
                               transforms.RandomHorizontalFlip(), transforms.ToTensor(), norm])
train_ds = datasets.ImageFolder("../data/imagenette2-320/train", transform=train_tf)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=4)
val_tf = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(227), transforms.ToTensor(), norm])
val_ds = datasets.ImageFolder("../data/imagenette2-320/val", transform=val_tf)
val_dl = DataLoader(val_ds, batch_size=128, num_workers=4)
# probe (uncomment to inspect the data)
# x, y = train_ds[0]
# print(f"train: {len(train_ds)} images | val: {len(val_ds)} images")
# print(f"labels: {len(train_ds.classes)} -> {train_ds.classes}")
# print(f"features per image: {tuple(x.shape)} = {x.numel():,} values | label of sample 0: {y}")

# trainig run
model = AlexNet()
cross_entropy = nn.CrossEntropyLoss()
model._init_weights()
model.to(device)
# SGD with momentum
optimizer = torch.optim.SGD(
    model.parameters(),
    lr = 0.01,
    momentum=0.9,
    weight_decay=0.0005,
)
# paper: divide LR by 10 when val stops improving
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.1, patience=2)
num_epochs = 10  # ~1000 iterations (74 batches per epoch)
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    for x, targets in train_dl:
        x, targets = x.to(device), targets.to(device)
        logits = model(x)
        loss = cross_entropy(logits, targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # validation, once per epoch
    model.eval()
    val_loss, correct = 0.0, 0
    with torch.no_grad():
        for x, targets in val_dl:
            x, targets = x.to(device), targets.to(device)
            logits = model(x)
            val_loss += cross_entropy(logits, targets).item()
            correct += (logits.argmax(1) == targets).sum().item()
    print(f"epoch: {epoch:2d} train loss: {train_loss / len(train_dl):.3f} "
          f"val loss: {val_loss / len(val_dl):.3f} val acc: {correct / len(val_ds):.3f}")
    sched.step(val_loss / len(val_dl))


epoch:  0 train loss: 3.753 val loss: 2.303 val acc: 0.099


KeyboardInterrupt: 